In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from utils import * 

# Load feature tables 

In [ ]:
df= pd.read_parquet('training_data/rdbms_training_data.parquet')
df_all_methods=pd.read_parquet('training_data/rdbms_all_methods_training_data.parquet')
FEATURES = ['num_agg_funcs', 'num_and_clauses', 'num_subqueries_with', 'num_eq_predicates', 'num_group_bys', 'num_join_occurrences', 'num_joins', 'num_unique_join_edges', 'num_order_bys', 'num_select_columns', 'num_where_clauses', 'num_qubits', 'width', 'depth', 'circuit_size', 'pauli_gate_count', 'two_qubit_gate_count', 'two_qubit_gate_percentage', 'locality_ratio', 'idling_score', 'density_score', 'statevector_saved_sparsity', 'statevector_saved_shannon_entropy', 'edge_count', 'max_degree', 'min_cut_upper', 'diameter', 'radius', 'average_degree', 'average_clustering_coefficient', 'average_shortest_path_length', 'central_point_of_dominance', 'std_dev_adjacency_matrix']
PRIMARY_KEY = "RowKey"

# Generated query

In [ ]:
# Plot the correlation matrix using utils.PlotCorrelationMatrix
from utils import PlotCorrelationMatrix
DROP_COLS = [
    "num_subqueries_with",
    "num_order_bys",
    "num_group_bys",
    "num_unique_join_edges",
    "num_join_occurrences",
    # "num_select_columns",
    # "num_where_clauses",
] # drop columns with low variance or non-meaningful

FEATURES = [c for c in FEATURES if c not in DROP_COLS]
pcm = PlotCorrelationMatrix(df, FEATURES, (8, 6))
# print nonunique columns
df[FEATURES].nunique().sort_values()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

METHODS = ["sqlite", "ducksql"]

# --------------------------------------------------
# BUILD COLUMN MAPS (cheap)
# --------------------------------------------------
time_cols = {
    m: [c for c in df.columns if c.endswith("time_s") and m in c]
    for m in METHODS
}
mem_cols = {
    m: [c for c in df.columns if c.endswith("mb") and m in c]
    for m in METHODS
}

# --------------------------------------------------
# VECTORIZED MIN PER METHOD
# --------------------------------------------------
time_min_df = pd.DataFrame({
    m: df[cols].min(axis=1)
    for m, cols in time_cols.items()
    if cols
})

mem_min_df = pd.DataFrame({
    m: df[cols].min(axis=1)
    for m, cols in mem_cols.items()
    if cols
})

# --------------------------------------------------
# BEST METHOD PER ROW (FAST)
# --------------------------------------------------
df["best_time_method"] = time_min_df.idxmin(axis=1)
df["best_mem_method"]  = mem_min_df.idxmin(axis=1)

# --------------------------------------------------
# COUNTS
# --------------------------------------------------
time_counts = (
    df["best_time_method"]
    .value_counts()
    .reindex(METHODS, fill_value=0)
)

mem_counts = (
    df["best_mem_method"]
    .value_counts()
    .reindex(METHODS, fill_value=0)
)

print("Best execution time counts:\n", time_counts)
print("\nBest memory usage counts:\n", mem_counts)

# --------------------------------------------------
# PLOT
# --------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

for ax, counts, title in zip(
    axes,
    [time_counts, mem_counts],
    ["Best Execution Time", "Best Memory Usage"],
):
    ax.barh(counts.index, counts.values)
    ax.set_title(f"{title} (Lower is Better)")
    ax.set_xlabel("Number of Rows")

    total = counts.sum()
    for i, c in enumerate(counts.values):
        pct = 100 * c / total if total else 0
        ax.text(c, i, f" {c} ({pct:.1f}%)", va="center")

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Merge Qiskit
df_with_qiskit = df.merge(qiskit_df, on=PRIMARY_KEY, suffixes=("", "_qiskit"))

RDBMS_METHODS = METHODS
QISKIT_METHODS = [
    "density_matrix",
    "matrix_product_state",
    "extended_stabilizer",
    # "statevector_saved",
    
    "statevector",
]
ALL_METHODS = RDBMS_METHODS + QISKIT_METHODS

# --------------------------------------------------
# CLEAN QISKIT ROWS (already efficient)
# --------------------------------------------------
rows_before = len(df_with_qiskit)

qiskit_cols = [
    c for c in df_with_qiskit.columns
    if any(m in c for m in QISKIT_METHODS)
]

df_with_qiskit = df_with_qiskit.dropna(subset=qiskit_cols)

rows_after = len(df_with_qiskit)
print(f"Rows before: {rows_before}")
print(f"Rows after:  {rows_after}")
print(f"Rows dropped: {rows_before - rows_after}")

# --------------------------------------------------
# FAST BEST-METHOD COMPUTATION
# --------------------------------------------------

# Build column maps (cheap)
time_cols = {
    m: [c for c in df_with_qiskit.columns if m in c and c.endswith(("time_s", "execution_time"))]
    for m in ALL_METHODS
}
mem_cols = {
    m: [c for c in df_with_qiskit.columns if m in c and c.endswith(("mb", "memory_usage"))]
    for m in ALL_METHODS
}

# Vectorized min-per-method
time_min_df = pd.DataFrame({
    m: df_with_qiskit[cols].min(axis=1)
    for m, cols in time_cols.items()
    if cols
})

mem_min_df = pd.DataFrame({
    m: df_with_qiskit[cols].min(axis=1)
    for m, cols in mem_cols.items()
    if cols
})

# Best method per row (vectorized)
df_with_qiskit["best_time_method"] = time_min_df.idxmin(axis=1)
df_with_qiskit["best_mem_method"]  = mem_min_df.idxmin(axis=1)

# --------------------------------------------------
# COUNTS
# --------------------------------------------------
time_counts = (
    df_with_qiskit["best_time_method"]
    .value_counts()
    .reindex(ALL_METHODS, fill_value=0)
)

mem_counts = (
    df_with_qiskit["best_mem_method"]
    .value_counts()
    .reindex(ALL_METHODS, fill_value=0)
)

print("Best execution time counts:\n", time_counts)
print("\nBest memory usage counts:\n", mem_counts)

# --------------------------------------------------
# PLOT (already fast enough)
# --------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharey=True)

for ax, counts, title in zip(
    axes,
    [time_counts, mem_counts],
    ["Best Execution Time", "Best Memory Usage"],
):
    ax.barh(counts.index, counts.values)
    ax.set_title(f"{title} (RDBMS + Qiskit)")
    ax.set_xlabel("Number of Rows")

    total = counts.sum()
    for i, c in enumerate(counts.values):
        pct = 100 * c / total if total else 0
        ax.text(c, i, f" {c} ({pct:.1f}%)", va="center")

plt.tight_layout()
plt.show()


In [ ]:
# Create a mask for rows that have valid data for ALL RDBMS methods
rdbms_cols_valid = []
for m in RDBMS_METHODS:
    rdbms_cols_valid.extend(time_cols.get(m, []))
    rdbms_cols_valid.extend(mem_cols.get(m, []))

df_rdbms_intersect = df_with_qiskit.dropna(subset=rdbms_cols_valid).copy()

print(f"Original rows: {len(df_with_qiskit)}")
print(f"Rows with all RDBMS methods (Intersection): {len(df_rdbms_intersect)}")

# --------------------------------------------------
# RECALCULATE BEST METHODS ON SUBSET
# --------------------------------------------------
time_min_intersect = pd.DataFrame({
    m: df_rdbms_intersect[cols].min(axis=1)
    for m, cols in time_cols.items() if cols
})

mem_min_intersect = pd.DataFrame({
    m: df_rdbms_intersect[cols].min(axis=1)
    for m, cols in mem_cols.items() if cols
})

df_rdbms_intersect["best_time_method"] = time_min_intersect.idxmin(axis=1)
df_rdbms_intersect["best_mem_method"]  = mem_min_intersect.idxmin(axis=1)

# --------------------------------------------------
# COUNTS
# --------------------------------------------------
time_counts_int = (
    df_rdbms_intersect["best_time_method"]
    .value_counts()
    .reindex(ALL_METHODS, fill_value=0)
)
mem_counts_int = (
    df_rdbms_intersect["best_mem_method"]
    .value_counts()
    .reindex(ALL_METHODS, fill_value=0)
)

print("Best execution time counts (Intersection):\n", time_counts_int)
print("\nBest memory usage counts (Intersection):\n", mem_counts_int)

# --------------------------------------------------
# PLOT
# --------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharey=True)

for ax, counts, title in zip(
    axes,
    [time_counts_int, mem_counts_int],
    ["Best Execution Time", "Best Memory Usage"],
):
    ax.barh(counts.index, counts.values)
    ax.set_title(f"{title} (Rows with all RDBMS data)")
    ax.set_xlabel("Number of Rows")

    total = counts.sum()
    for i, c in enumerate(counts.values):
        pct = 100 * c / total if total else 0
        ax.text(c, i, f" {c} ({pct:.1f}%)", va="center")

plt.tight_layout()
plt.show()

In [ ]:
df_all_methods

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Merge Qiskit
df_with_qiskit = df_all_methods.merge(qiskit_df, on=PRIMARY_KEY, suffixes=("", "_qiskit"))

RDBMS_METHODS = METHODS
QISKIT_METHODS = [
    "density_matrix",
    "matrix_product_state",
    "extended_stabilizer",
    # "statevector_saved",
    
    "statevector",
]
ALL_METHODS = RDBMS_METHODS + QISKIT_METHODS

# --------------------------------------------------
# CLEAN QISKIT ROWS (already efficient)
# --------------------------------------------------
rows_before = len(df_with_qiskit)

qiskit_cols = [
    c for c in df_with_qiskit.columns
    if any(m in c for m in QISKIT_METHODS)
]

df_with_qiskit = df_with_qiskit.dropna(subset=qiskit_cols)

rows_after = len(df_with_qiskit)
print(f"Rows before: {rows_before}")
print(f"Rows after:  {rows_after}")
print(f"Rows dropped: {rows_before - rows_after}")

# --------------------------------------------------
# FAST BEST-METHOD COMPUTATION
# --------------------------------------------------

# Build column maps (cheap)
time_cols = {
    m: [c for c in df_with_qiskit.columns if m in c and c.endswith(("time_s", "execution_time"))]
    for m in ALL_METHODS
}
mem_cols = {
    m: [c for c in df_with_qiskit.columns if m in c and c.endswith(("mb", "memory_usage"))]
    for m in ALL_METHODS
}

# Vectorized min-per-method
time_min_df = pd.DataFrame({
    m: df_with_qiskit[cols].min(axis=1)
    for m, cols in time_cols.items()
    if cols
})

mem_min_df = pd.DataFrame({
    m: df_with_qiskit[cols].min(axis=1)
    for m, cols in mem_cols.items()
    if cols
})

# Best method per row (vectorized)
df_with_qiskit["best_time_method"] = time_min_df.idxmin(axis=1)
df_with_qiskit["best_mem_method"]  = mem_min_df.idxmin(axis=1)

# --------------------------------------------------
# COUNTS
# --------------------------------------------------
time_counts = (
    df_with_qiskit["best_time_method"]
    .value_counts()
    .reindex(ALL_METHODS, fill_value=0)
)

mem_counts = (
    df_with_qiskit["best_mem_method"]
    .value_counts()
    .reindex(ALL_METHODS, fill_value=0)
)

print("Best execution time counts:\n", time_counts)
print("\nBest memory usage counts:\n", mem_counts)

# --------------------------------------------------
# PLOT (already fast enough)
# --------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharey=True)

for ax, counts, title in zip(
    axes,
    [time_counts, mem_counts],
    ["Best Execution Time", "Best Memory Usage"],
):
    ax.barh(counts.index, counts.values)
    ax.set_title(f"{title} (RDBMS + Qiskit)")
    ax.set_xlabel("Number of Rows")

    total = counts.sum()
    for i, c in enumerate(counts.values):
        pct = 100 * c / total if total else 0
        ax.text(c, i, f" {c} ({pct:.1f}%)", va="center")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

METHODS = ["sqlite", "ducksql"]

# --------------------------------------------------
# BUILD COLUMN MAPS (cheap)
# --------------------------------------------------
time_cols = {
    m: [c for c in df_all_methods.columns if c.endswith("time_s") and m in c]
    for m in METHODS
}
mem_cols = {
    m: [c for c in df_all_methods.columns if c.endswith("mb") and m in c]
    for m in METHODS
}

# --------------------------------------------------
# VECTORIZED MIN PER METHOD
# --------------------------------------------------
time_min_df = pd.DataFrame({
    m: df_all_methods[cols].min(axis=1)
    for m, cols in time_cols.items()
    if cols
})

mem_min_df = pd.DataFrame({
    m: df_all_methods[cols].min(axis=1)
    for m, cols in mem_cols.items()
    if cols
})

# --------------------------------------------------
# BEST METHOD PER ROW (FAST)
# --------------------------------------------------
df_all_methods["best_time_method"] = time_min_df.idxmin(axis=1)
df_all_methods["best_mem_method"]  = mem_min_df.idxmin(axis=1)

# --------------------------------------------------
# COUNTS
# --------------------------------------------------
time_counts = (
    df_all_methods["best_time_method"]
    .value_counts()
    .reindex(METHODS, fill_value=0)
)

mem_counts = (
    df_all_methods["best_mem_method"]
    .value_counts()
    .reindex(METHODS, fill_value=0)
)

print("Best execution time counts:\n", time_counts)
print("\nBest memory usage counts:\n", mem_counts)

# --------------------------------------------------
# PLOT
# --------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

for ax, counts, title in zip(
    axes,
    [time_counts, mem_counts],
    ["Best Execution Time", "Best Memory Usage"],
):
    ax.barh(counts.index, counts.values)
    ax.set_title(f"{title} (Lower is Better)")
    ax.set_xlabel("Number of Rows")

    total = counts.sum()
    for i, c in enumerate(counts.values):
        pct = 100 * c / total if total else 0
        ax.text(c, i, f" {c} ({pct:.1f}%)", va="center")

plt.tight_layout()
plt.show()


# Multiclass Qiskit and RDBMS methods

In [ ]:
df_with_qiskit[FEATURES].info()

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from utils import PlotConfusionReports

ALL_METHODS = QISKIT_METHODS + ["sqlite"]

# --- data ---
X, y = df_with_qiskit[FEATURES], df_with_qiskit["best_mem_method"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

# --- label encoding + class weights ---
le = LabelEncoder()
y_train_enc, y_test_enc = le.fit_transform(y_train), le.transform(y_test)
cw_enc = dict(zip(np.unique(y_train_enc), compute_class_weight("balanced", classes=np.unique(y_train_enc), y=y_train_enc)))
sw_xgb = np.array([cw_enc[c] for c in y_train_enc])

# --- models ---
models = {
    "Random Forest": RandomForestClassifier(300, class_weight="balanced", n_jobs=-1, random_state=42),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "Logistic Regression": Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))]),
    "Linear SVM": Pipeline([("scaler", StandardScaler()), ("clf", SVC(kernel="linear", class_weight="balanced"))]),
    "KNN": Pipeline([("scaler", StandardScaler()), ("clf", KNeighborsClassifier(7))]),
    "XGBoost": XGBClassifier(300, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
                             objective="multi:softmax", num_class=len(le.classes_), eval_metric="mlogloss", random_state=42)
}

# --- train / predict ---
preds = {}
for name, model in models.items():
    if name == "XGBoost":
        model.fit(X_train, y_train_enc, sample_weight=sw_xgb)
        y_pred = le.inverse_transform(model.predict(X_test))
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    preds[name] = y_pred
    print(f"{name:20s} accuracy = {accuracy_score(y_test, y_pred):.3f}")

# --- confusion plots ---
PlotConfusionReports(y_test, preds, ALL_METHODS)


# RDBMS vs Qiskit BINARY classification

In [ ]:
from sklearn.metrics import classification_report
# --- binary target ---
RDBMS_METHODS = {"psql", "ducksql", "sqlite", "infiniquantum"}
QISKIT_METHODS = {"statevector_saved", "density_matrix", "matrix_product_state", "extended_stabilizer", "statevector_saved"}

df_bin = df_with_qiskit.copy()
df_bin["family"] = df_bin["best_mem_method"].apply(
    lambda m: "rdbms" if m in RDBMS_METHODS else "qiskit" if m in QISKIT_METHODS else np.nan
)
df_bin = df_bin.dropna(subset=["family"])

X, y = df_bin[FEATURES], df_bin["family"]
print("Class distribution:\n", y.value_counts(), "\n")

# --- split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- encoding + weights ---
le = LabelEncoder()
y_train_enc, y_test_enc = le.fit_transform(y_train), le.transform(y_test)

cw = dict(zip(
    np.unique(y_train),
    compute_class_weight(class_weight="balanced",
                         classes=np.unique(y_train),
                         y=y_train)
))

cw_enc = dict(zip(
    np.unique(y_train_enc),
    compute_class_weight(class_weight="balanced",
                         classes=np.unique(y_train_enc),
                         y=y_train_enc)
))

sample_weights_xgb = np.array([cw_enc[c] for c in y_train_enc])

# --- models ---
models = {
    "K-Nearest Neighbors": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", KNeighborsClassifier(7))
    ]),
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))
    ]),
    "Linear SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(kernel="linear", class_weight="balanced"))
    ]),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(300, n_jobs=-1, class_weight="balanced", random_state=42),
    "XGBoost Classifier": XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        objective="binary:logistic", eval_metric="logloss",
        random_state=42
    )
}

# --- train / evaluate ---
model_to_y_pred = {}

for name, model in models.items():
    if name == "XGBoost Classifier":
        model.fit(X_train, y_train_enc, sample_weight=sample_weights_xgb)
        y_pred = le.inverse_transform((model.predict(X_test) > 0.5).astype(int))
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

    model_to_y_pred[name] = y_pred
    print(f"{name:20s} accuracy = {accuracy_score(y_test, y_pred):.3f}")
    print(classification_report(y_test, y_pred, digits=3))
    print("-" * 50)

# --- confusion matrices ---
PlotConfusionReports(y_test, model_to_y_pred, methods=["rdbms", "qiskit"])


# Analysing importance of SQL features for RDBMS profiling

In [ ]:
# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt

models_to_compare = {
    "Linear SVM": models["Linear SVM"],
    # "Logistic Regression": models["Logistic Regression"],
    "Random Forest": models["Random Forest"],
    "XGBoost Classifier": models["XGBoost Classifier"],
}

# TOP_K = 10
# fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# for ax, (name, pipeline) in zip(axes, models_to_compare.items()):

#     clf = pipeline.named_steps["clf"]
#     coefs = clf.coef_.ravel()

#     importance_df = pd.DataFrame({
#         "feature": FEATURES,
#         "coefficient": coefs,
#         "abs_importance": np.abs(coefs)
#     }).sort_values("abs_importance", ascending=False)

#     print(f"\nTop 15 most important features ({name}):\n")
#     print(importance_df.head(15))

#     plot_df = importance_df.head(TOP_K).iloc[::-1]

#     ax.barh(plot_df["feature"], plot_df["coefficient"])
#     ax.axvline(0, linestyle="--", linewidth=1)
#     ax.set_title(f"{name} Feature Importance")
#     ax.set_xlabel("Coefficient Value")
#     ax.set_ylabel("Feature")

# fig.suptitle(
#     "Linear Model Feature Importance\n(Sign indicates RDBMS ↔ Qiskit decision)",
#     fontsize=14
# )

# plt.tight_layout(rect=[0, 0, 1, 0.95])
# plt.show()

# print("\nInterpretation:")
# print("• Positive coefficient  → pushes prediction toward class 'qiskit'")
# print("• Negative coefficient  → pushes prediction toward class 'rdbms'")
# print("• Larger magnitude      → stronger influence on decision")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

TOP_K = 10
fig, axes = plt.subplots(1, len(models_to_compare), figsize=(16, 6))
axes = axes.ravel()

for ax, (name, model) in zip(axes, models_to_compare.items()):
    clf = model.named_steps["clf"] if hasattr(model, "named_steps") else model

    # Linear models: coefficients
    if hasattr(clf, "coef_"):
        imp = clf.coef_.ravel()
        feat_names = FEATURES
    # Tree models: feature_importances_
    elif hasattr(clf, "feature_importances_"):
        imp = clf.feature_importances_
        feat_names = FEATURES

    # Build DataFrame and select top K features
    df = pd.DataFrame({
        "feature": feat_names,
        "importance": imp,
        "abs_importance": np.abs(imp)
    }).sort_values("abs_importance", ascending=False).head(TOP_K).iloc[::-1]

    ax.barh(df["feature"], df["importance"])  # default color
    if hasattr(clf, "coef_"):
        ax.axvline(0, linestyle="--", linewidth=1)
    ax.set_title(f"{name} Feature Importance")
    ax.set_xlabel("Importance")
    ax.set_ylabel("Feature")

    print(f"\nTop {TOP_K} features ({name}):\n", df)

fig.suptitle("Feature Importance (Linear coef or Tree importance)", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


In [ ]:
# import matplotlib.pyplot as plt

# fig, axes = plt.subplots(2, 1, figsize=(9, 10))

# # Histogram: num_and_clauses
# axes[0].hist(df_bin.loc[df_bin.family == "rdbms",  "num_and_clauses"],
#              bins=10, density=True, alpha=0.6, label="RDBMS")
# axes[0].hist(df_bin.loc[df_bin.family == "qiskit", "num_and_clauses"],
#              bins=10, density=True, alpha=0.6, label="Qiskit")
# axes[0].set_xlabel("num_and_clauses")
# axes[0].set_ylabel("Density")
# axes[0].set_title("num_and_clauses by execution family")
# axes[0].legend()

# # Scatter: num_joins vs num_agg_funcs
# axes[1].scatter(df_bin.loc[df_bin.family == "rdbms", "num_joins"],
#                 df_bin.loc[df_bin.family == "rdbms", "num_agg_funcs"],
#                 alpha=0.6, label="RDBMS")
# axes[1].scatter(df_bin.loc[df_bin.family == "qiskit", "num_joins"],
#                 df_bin.loc[df_bin.family == "qiskit", "num_agg_funcs"],
#                 alpha=0.6, label="Qiskit")
# axes[1].set_xlabel("num_joins")
# axes[1].set_ylabel("num_agg_funcs")
# axes[1].set_title("num_joins vs num_agg_funcs")
# axes[1].legend()

# plt.tight_layout()
# plt.show()
